In [ ]:
# Célula 0 — Tokens, Drive e código do repositório
import os
from google.colab import drive, userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # opcional, acelera downloads
except Exception:
    print("HF_TOKEN não configurado (ok, só fica mais lento).")

drive.mount("/content/drive")

REPO = "/content/experimentos-llm-local"
if not os.path.exists(REPO):
    !git clone https://github.com/Arthur-galvao/experimentos-llm-local.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}

# F1a: autodestilação do Dory L2 em cascata

Notebook fino: a lógica está em `scripts/f1a_self_distill.py`. Tudo retoma de onde parou se a sessão cair (perde no máximo 1 lote de 64 problemas).
1. Baseline do L2 na `custom_v4`: só os 42 hard novos geram, o easy e o medium vêm do cache da v3.
2. F1a em três estágios, do mais barato ao mais caro. Um problema resolvido num estágio não entra nos seguintes:
   - `nothink`: 4 amostras sem raciocínio. Quem acerta as 4 vira exemplo de CoT vazia.
   - `efficient`: 8 amostras no modo de raciocínio curto do template do Dory.
   - `full`: 8 amostras com raciocínio completo, só nos problemas que sobraram. É o estágio caro.

Cada estágio imprime o teto de custo antes de gerar e, a cada lote, as horas de GPU e as CU que faltam. O SFT (`data/short_cot_sft.jsonl`) é remontado ao fim de cada estágio, então dá para parar depois de qualquer um. Todas as amostras, inclusive as erradas, ficam em `data/f1a/samples_*.jsonl` para o SFT arbóreo.

In [ ]:
# Célula 1 — Hardware e instalação (vLLM da PR #60149 + dependências)
# 1ª vez: gera as wheels e guarda no Drive (MyDrive/coder-comprimido/wheels), demora como antes.
# Depois: instala só das wheels do Drive, sem clonar nem compilar. Para atualizar o vLLM, apague essa pasta.
!bash scripts/colab_install.sh

In [ ]:
# Célula 2 — Baseline do L2 no hard novo (custom_v4)
!python scripts/f0_run_baseline.py --target L2 --scope custom_quick

In [ ]:
# Célula 3 — F1a, estágios baratos: sem raciocínio e raciocínio curto.
# N_PROBLEMS pode subir depois: os primeiros problemas continuam os mesmos e só os novos geram.
N_PROBLEMS = 500
!python scripts/f1a_self_distill.py --stage nothink --n-problems {N_PROBLEMS}
!python scripts/f1a_self_distill.py --stage efficient --n-problems {N_PROBLEMS}

In [ ]:
# Célula 4 — F1a, estágio caro: raciocínio completo só nos problemas que os estágios anteriores não resolveram.
# Olhe o teto de custo que ele imprime antes do primeiro lote; dá para interromper e retomar depois.
!python scripts/f1a_self_distill.py --stage full --n-problems {N_PROBLEMS}